# 05 - Treino Final e Exportação

Reajuste do melhor modelo e salvamento em `models/`.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, classification_report

data = pd.read_parquet("../data/processed/flights_sample.parquet")
features = ["AIRLINE","ORIGIN_AIRPORT","DESTINATION_AIRPORT","MONTH","DAY_OF_WEEK","DEP_HOUR","DISTANCE","IS_WEEKEND"]
X = data[features]
y = data["DELAYED"]

cat_features = ["AIRLINE","ORIGIN_AIRPORT","DESTINATION_AIRPORT"]
num_features = ["MONTH","DAY_OF_WEEK","DEP_HOUR","DISTANCE","IS_WEEKEND"]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore"))]), cat_features),
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_features)
])

rf = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=150, max_depth=18, random_state=42, n_jobs=-1, class_weight="balanced"
    ))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
rf.fit(X_train, y_train)
probas = rf.predict_proba(X_test)[:,1]
auc = roc_auc_score(y_test, probas)
prec, rec, thresh = precision_recall_curve(y_test, probas)
f1_scores = 2 * prec * rec / (prec + rec + 1e-9)
best_idx = f1_scores.argmax()
best_threshold = thresh[best_idx]
print(f"ROC-AUC holdout: {auc:.3f}")
print(f"Melhor threshold (F1 classe 1): {best_threshold:.3f}")
y_pred_thresh = (probas >= best_threshold).astype(int)
print(classification_report(y_test, y_pred_thresh))

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)
out_path = models_dir / "random_forest_delay_model.pkl"
joblib.dump({"model": rf, "threshold": float(best_threshold)}, out_path)
print(f"Modelo salvo em {out_path} (inclui threshold)")


ROC-AUC holdout: 0.647
Melhor threshold (F1 classe 1): 0.487
              precision    recall  f1-score   support

           0       0.88      0.55      0.68     49323
           1       0.24      0.67      0.36     10677

    accuracy                           0.57     60000
   macro avg       0.56      0.61      0.52     60000
weighted avg       0.77      0.57      0.62     60000

Modelo salvo em ..\models\random_forest_delay_model.pkl (inclui threshold)
